In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reproduce.py").is_file())
os.chdir(ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
   
PROJECT_PATH = Path("data/genomics")

In [ ]:
meta = pd.read_csv(PROJECT_PATH / "reference/metadata_complete.csv")
meta['population'] = meta['Strain'] + '_' + meta['Culture'].astype(str).str.zfill(2)
meta

In [ ]:
dfs = []

for i, row in meta.iterrows():
    df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
    df["Strain"] = row['Strain']
    df["Culture"] = row['Culture']
    df["Day"] = int(row['Day'])
    dfs.append(df)

df = pd.concat(dfs)
df

In [ ]:
select_lineage = "PC"
pops = meta.query(f'Strain=="{select_lineage}"')['population'].unique()
timepoints = sorted(meta.query(f'Strain=="{select_lineage}"')['Day'].unique())
print(pops)
print(timepoints)

In [ ]:
meta.query(f'population=="{select_lineage}_01"')

In [ ]:
def traceAlleleFreq(pop, min_freq=0.33):

    D = []
    sorted_meta = meta.query(f'population=="{pop}"').sort_values(by='Day')
    for i, row in sorted_meta.iterrows():
        df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
        D.append(df)

    # Remove mutations detected in the wild-type background
    # 1. Select background mutations with strong signal
    wt_background=D[0].loc[D[0].frequency>0.01, 'position']
    #print(f"WT background mutations: {wt_background.values}")
    D_=[]

    #print(f"# of mutations\t# of (freq>{min_freq}):")
    for i in range(len(D)):
        # 2. Filter out mutations by position on the chromosome
        df=D[i][~D[i]['position'].isin(wt_background)]
        D_.append(df)
        #print(f"{df.shape[0]}\t{sum(df['frequency']>min_freq)}")
    
    tracked_positions=[]
    for i in range(len(D_)):
        d=D_[i]
        tracked_positions.extend(list(d.loc[ (d['frequency']>min_freq), 'position' ])) #& ~d['mutation_category'].isin(['mobile_element_insertion','large_deletion']), 'position' ] ))
    tracked_positions=set(tracked_positions)
    print(f"Population {pop} # of tracked mutations: {len(tracked_positions)}")

    ## 3. Trace the frequency of those mutations
    T=[]
    for pos in tracked_positions:

        freq=[]
        for i in range(len(D_)):
            d = D_[i]
            ind = d['position']==pos
            
            if sum(ind)<1:
                freq.append(0)
            else:
                freq.append( (d.loc[ind , 'frequency'].values)[0] )
                gene_name = d.loc[ind, 'gene_name'].values[0]
                gene_product = d.loc[ind, 'gene_product'].values[0]
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
                aa_new_seq = d.loc[ind, 'aa_new_seq'].values[0]
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
                new_seq = d.loc[ind, 'new_seq'].values[0]
                codon_ref_seq = d.loc[ind, 'codon_ref_seq'].values[0]
                aa_pos = d.loc[ind, 'aa_position'].values[0]
                gene_pos = d.loc[ind, 'gene_position'].values[0]
                mut_cat = d.loc[ind, 'mutation_category'].values[0]

        T.append({'position': pos, 'freq': np.round(freq,2), 
                'gene_name':gene_name, 'gene_product':gene_product,
                'aa_ref_seq':aa_ref_seq, 'aa_new_seq':aa_new_seq,
                'aa_pos': aa_pos, 'gene_pos':gene_pos, 'mut_cat': mut_cat,
                'new_seq': new_seq, 'codon_ref_seq': codon_ref_seq})

    T=pd.DataFrame(T)
    T.fillna({'aa_ref_seq': '',
              'aa_new_seq': '',
              'aa_pos': ''},inplace=True)
    
    nsi = T['aa_pos']==''
    T.loc[nsi,'label'] = T.loc[nsi,'gene_name'] + ' ' + T.loc[nsi, 'mut_cat']
    T.loc[~nsi, 'label'] = T.loc[~nsi,'gene_name'] + ' ' + T.loc[~nsi,'aa_ref_seq'] + T.loc[~nsi, 'aa_pos'].astype(str).str.replace(r'\.0','',regex=True) + T.loc[~nsi,'aa_new_seq']
        # T.loc[nsi,'gene_pos'] + ' '
    T.sort_values(by=['mut_cat','gene_name'],ascending=False,inplace=True)
    T.reset_index(inplace=True,drop=True)
    
    return T

In [ ]:
af=[]
for pop in pops:
    # print(traceAlleleFreq(pop, min_freq=0.2).shape)
    af.append(traceAlleleFreq(pop, min_freq=0.1))
    af[-1].to_csv(f"data/genomics/data/processed/traced_alleles/{select_lineage}/{pop}.csv", index=False)

In [ ]:
clt = 4

selB_rows = af[clt-1].query('gene_name=="phoQ"')

# Display the filtered rows
selB_rows

In [ ]:
af[3].loc[
    (af[3]['gene_name'] == 'phoQ') & (af[3]['gene_pos'] == 'coding (1265-1268/1461 nt)'),
    'label'
] = 'phoQ IS2 insertion'
#
af[3].loc[
    (af[3]['gene_name'] == 'phoQ') & (af[3]['gene_pos'] == 'coding (136-140/1461 nt)'),
    'label'
] = 'phoQ IS5 insertion'

In [ ]:
pd.set_option("display.max_rows", 100)
af[3].head(100)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Concatenate unique labels and count occurrences
all_labels = np.concatenate([df.label.unique() for df in af])
unique_labels, counts = np.unique(all_labels, return_counts=True)

# Create a DataFrame with labels and counts
label_counts = pd.DataFrame({'label': unique_labels, 'count': counts})

# Map gene_name and freq (last value) to labels
gene_name_map = {row['label']: row['gene_name'] for df in af for _, row in df.iterrows()}
freq_map = {row['label']: row['freq'][-1] for df in af for _, row in df.iterrows()}  # Extract the last value from 'freq'
label_counts['gene_name'] = label_counts['label'].map(gene_name_map)
label_counts['freq'] = label_counts['label'].map(freq_map)

# Separate labels by gene_name
prs_labels = label_counts[label_counts['gene_name'] == 'prs']
phoQ_labels = label_counts[label_counts['gene_name'] == 'phoQ']
other_labels = label_counts[(label_counts['gene_name'] != 'prs') & (label_counts['gene_name'] != 'phoQ')]

# Sort `prs` and `phoQ` groups by frequency (just in case there's more than one)
prs_labels = prs_labels.sort_values(by='freq', ascending=True)
phoQ_labels = phoQ_labels.sort_values(by='freq', ascending=True)

# Sort other labels by last freq value
other_labels = other_labels.sort_values(by='freq', ascending=False)

# Combine the groups: prs first, phoQ second, others last
label_counts_sorted = pd.concat([prs_labels, phoQ_labels, other_labels], ignore_index=True)

# Create a categorical order for labels
label_order = label_counts_sorted['label'].tolist()
label_counts_sorted['label'] = pd.Categorical(
    label_counts_sorted['label'], categories=label_order, ordered=True
)

# Define manual colors for each label
label_color_map = {
    'prs A114V': '#1f77b4',  # Blue
    'corA H75N': '#ff7f0e',  # Orange
    'cysS Y298S': '#2ca02c',  # Green
    'mrcB SNP': '#d62728',  # Red
    'mgtL/mgtA indel': '#9467bd',  # Purple
    'pheS A14P': '#8c564b',  # Brown
    'phoQ G39C': '#e377c2',  # Pink
    'phoQ IS5 insertion': '#7f7f7f',  # Gray
    'phoQ IS2 insertion': 'blue',  # Gray
    'phoQ indel': '#bcbd22',  # Olive
    'prs A76V': '#17becf',  # Cyan
    'prs I151T': '#aec7e8',  # Light Blue
    'prs R79C': '#ffbb78',  # Light Orange
    'rbsA S157R': '#98df8a',  # Light Green
    'relB P45S': '#ff9896',  # Light Red
    'waaF D13E': '#c5b0d5',  # Light Purple
    'yicC IS2 insertion': '#c49c94',  # Light Brown
    'yjbB L14Q': '#f7b6d2'  # Light Pink
}

# Define partial alternate labels
partial_alternate_label_map = {
    'mrcB|mrcB T|T702|657S|S': 'mrcB SNP',
    'mgtL/mgtA small_indel': 'mgtL/mgtA indel',
    'phoQ small_indel': 'phoQ indel',
    'yicC mobile_element_insertion': 'yicC IS2 insertion',
    # Add more alternate labels as needed
}

# Add alternate labels to the DataFrame (fallback to original labels)
label_counts_sorted['alternate_label'] = label_counts_sorted['label'].map(
    partial_alternate_label_map
).fillna(label_counts_sorted['label'])  # Retain original labels if not in the map

# Apply colors to the barplot
fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(
    x='alternate_label',  # Use alternate labels for x-axis
    y='count', 
    data=label_counts_sorted, 
    palette=label_color_map,  # Use the custom color mapping
    ax=ax
)

# Customize y-axis limits and ticks
ax.set_ylim(0, 4)  # Set y-axis limits
ax.set_yticks([0, 1, 2, 3, 4])  # Set y-axis tick locations
# Add text above each bar
for i, row in label_counts_sorted.iterrows():
    bar_x = i  # x-coordinate of the bar
    bar_height = row['count']  # y-coordinate of the top of the bar
    freq_value = row['freq']  # Text to display (last freq value)
    ax.text(
        bar_x, 
        bar_height + 0.2,  # Slightly above the bar
        f"{freq_value:.1f}",  # Format frequency to 2 decimal places
        ha='center', 
        va='bottom', 
        fontsize=15
    )

# Customize tick and font size
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
ax.set_xlabel("Mutations", fontsize=20)
ax.set_ylabel("Number of Cultures", fontsize=20)
ax.set_title("High Frequency Mutations of MG$^{{\\mathrm{{CEF}}}}$", fontsize=20)


# # Save the figure
# output_path = PROJECT_PATH / "figures/mutation_counts/PCmutations_prs_phoQ_rest_by_freq_with_colors.png"
# fig.savefig(output_path, dpi=300, bbox_inches='tight')


In [ ]:
tracked_mut = af[4].query(f'label.isin({list(select_mutations)})')
#tracked_mut = tracked_mut.set_index('label').loc[select_mutations,:].reset_index()
tracked_mut

In [ ]:
all = []
for ix, df in enumerate(af):
    dff = df.copy()
    dff['Pop'] = ix+1 
    all.append(dff)
combined_df=pd.concat(all)


combined_df['last_freq'] = combined_df['freq'].apply(lambda x: x[-1])
combined_df

In [ ]:
prs_rows = combined_df[combined_df['gene_name'] == 'prs']

# Print the subsetted rows
prs_rows

In [ ]:
def get_mutation_group_and_labels(group):
    # """
    # Returns select mutations and alternate labels based on the specified group.
    
    # Parameters:
    #     group (str): The group to retrieve mutations for. Options are 'prs_phoQ' or 'other'.
    
    # Returns:
    #     tuple: A tuple containing a list of selected mutations and a dictionary of alternate labels.
    # """
    # Select mutations based on group
    if group == 'prs_phoQ':
        select_mutations = [
            'prs A114V', 'prs R79C', 'prs A11S', 'prs A76V', 'prs I151T',
            'phoQ G39C', 'phoQ IS2 insertion', 'phoQ small_indel',  'phoQ IS5 insertion',
        ]
        alt_labels = {
            'phoQ small_indel': 'phoQ indel',
        }
        manual_label_order = [
            'prs A114V', 'prs R79C', 'prs A76V', 'prs A11S',  'prs I151T',
            'phoQ G39C', 'phoQ IS2 insertion', 'phoQ small_indel',  'phoQ IS5 insertion',
]
    elif group == 'other':
        select_mutations = [
            'waaF D13E', 'cysS Y298S', 'mrcB|mrcB T|T702|657S|S', 'pheS A14P',
            'relB P45S', 'yjbB L14Q', 'rbsA S157R', 'ygfB P184Q', 'aceK V471G',
            'yicC mobile_element_insertion', 'corA H75N', 'glnH A17V', 'pitA T184T',
            'corA L50Q', 'mgtL/mgtA small_indel', 'nadK T200P', 'eutH P393Q',
            'manZ/yobD snp_intergenic', 'cstA A392S', 'pgaC I94I', 'cra S18N',
            'yhfG/ppiA snp_intergenic', 'pgaC I81L', 'yaaA I52T',
            'yhaC/rnpB snp_intergenic', 'marR small_indel', 'abgT L394F', 'phoP R67H',
            'yaaA S53I', 'ytfJ A73E', 'mgtT Y23S', 'marC/marR small_indel',
            'marR Q117*', 'insH1/mmuP snp_intergenic', 'yfbS A38S',
            'insH5/lomR snp_intergenic', 'obgE Q71K', 'rapA W549*', 'gsk F3I',
            'rlmB D59E', 'yiiG/frvR snp_intergenic', 'bamD P32Q', 'bioH S225C'
        ]
        alt_labels = {
            'mrcB|mrcB T|T702|657S|S': 'mrcbB T702S',
            'yicC mobile_element_insertion': 'yicC IS insertion',
            'mgtL/mgtA small_indel': 'mgtL/mgtA indel',
            'manZ/yobD snp_intergenic': 'manZ/yobD snp intergenic',
            'yhfG/ppiA snp_intergenic': 'yhfG/ppiA snp intergenic',
            'yhaC/rnpB snp_intergenic': 'yhaC/rnpB snp intergenic',
            'marR small_indel': 'marR indel',
            'marC/marR small_indel': 'marC/marR indel',
            'marR Q117*': 'marR Q117*',
            'insH1/mmuP snp_intergenic': 'insH1/mmuP snp intergenic',
            'insH5/lomR snp_intergenic': 'insH5/lomR snp intergenic',
            'rapA W549*': 'rapA W549*',
            'yiiG/frvR snp_intergenic': 'yiiG/frvR snp intergenic'
        }
        manual_label_order = [
            'waaF D13E', 'cysS Y298S', 'mrcB|mrcB T|T702|657S|S', 'pheS A14P',
            'relB P45S', 'yjbB L14Q', 'rbsA S157R', 'ygfB P184Q', 'aceK V471G',
            'yicC mobile_element_insertion', 'corA H75N', 'glnH A17V', 'pitA T184T',
            'corA L50Q', 'mgtL/mgtA small_indel', 'nadK T200P', 'eutH P393Q',
            'manZ/yobD snp_intergenic', 'cstA A392S', 'pgaC I94I', 'cra S18N',
            'yhfG/ppiA snp_intergenic', 'pgaC I81L', 'yaaA I52T',
            'yhaC/rnpB snp_intergenic', 'marR small_indel', 'abgT L394F', 'phoP R67H',
            'yaaA S53I', 'ytfJ A73E', 'mgtT Y23S', 'marC/marR small_indel',
            'marR Q117*', 'insH1/mmuP snp_intergenic', 'yfbS A38S',
            'insH5/lomR snp_intergenic', 'obgE Q71K', 'rapA W549*', 'gsk F3I',
            'rlmB D59E', 'yiiG/frvR snp_intergenic', 'bamD P32Q', 'bioH S225C'
]
    else:
        raise ValueError("Invalid group. Options are 'prs_phoQ' or 'other'.")

    return select_mutations, alt_labels, manual_label_order


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
plt.rcParams["font.family"] = "Nimbus Roman"
# Get mutation group for 'prs_phoQ'
group = 'prs_phoQ'
select_mutations, alt_labels, manual_label_order = get_mutation_group_and_labels(group)

# Get the 'Greens' colormap and customize it
standard_map = plt.cm.get_cmap('Greens')
new_colors = standard_map(np.linspace(0, 1, 256))
new_colors[0] = np.array([1, 1, 1, 1])  # Replace the first color (for 0 values) with white
custom_map = mcolors.ListedColormap(new_colors)

# Ensure the 'Pop' column exists in all DataFrames in af
for idx, df in enumerate(af):
    if 'Pop' not in df.columns:
        df['Pop'] = idx + 1  # Assign a unique culture number based on the index

# Map gene_name and the last freq value to labels
gene_name_map = {row['label']: row['gene_name'] for df in af for _, row in df.iterrows()}
freq_map = {
    (row['label'], row['Pop']): row['freq'][-1] if isinstance(row['freq'], (list, np.ndarray)) else row['freq']
    for df in af for _, row in df.iterrows()
}

# Add 'last_freq' column to each DataFrame in af
for df in af:
    df['last_freq'] = df.apply(lambda row: freq_map.get((row['label'], row['Pop']), 0), axis=1)

# Combine all DataFrames into one
all_data = pd.concat(af, ignore_index=True)

# Filter for selected mutations
filtered_data = all_data[all_data['label'].isin(select_mutations)]

# Pivot the DataFrame to have 'label' as rows, 'Pop' as columns, and 'last_freq' as values
heatmap_data = filtered_data.pivot_table(index='label', columns='Pop', values='last_freq', fill_value=0)

# Reindex the rows to match the manual label order
heatmap_data = heatmap_data.reindex(manual_label_order)

# Define the full set of cultures
all_cultures = range(1, 11)  # Assuming cultures are numbered 1 through 10

# Reindex the columns to include all cultures
heatmap_data = heatmap_data.reindex(columns=all_cultures, fill_value=0)

# Convert the DataFrame to a numpy array for plotting
freq_mat = heatmap_data.values
timepoints = heatmap_data.columns

# Define desired cell size (in inches)
cell_width = 0.7  # Width of each cell

# Calculate figure size
figsize = ((9 * cell_width), 7)

# Create a figure and axes with the calculated size
fig, ax = plt.subplots(figsize=figsize)
annot_matrix = np.where(freq_mat.T == 0, '', freq_mat.T.round(1).astype(str))

# Apply alternate labels for the plot
plot_labels = [alt_labels.get(label, label) for label in manual_label_order]

# Plot the heatmap
sns.heatmap(
    freq_mat.T,
    yticklabels=timepoints,
    xticklabels=False,
    cmap=custom_map,
    vmin=0,
    vmax=1,
    ax=ax,
    annot=annot_matrix,
    fmt="",
    annot_kws={'size': 20, 'ha': 'center', 'va': 'center'}, cbar=False,
    linewidths=0.5,
    linecolor='lightgrey'
)

# Customize plot labels
ax.set_ylabel('Culture Number', fontsize=25)
ax.set_xlabel('', fontsize=25)
ax.tick_params(axis='both', which='major', labelsize=15, length=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor', fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

# Set plot spines
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
    spine.set_color("black")
plt.subplots_adjust(bottom=0.1)

# Display the plot
plt.tight_layout()
plt.show()
fig.savefig(PROJECT_PATH / "figures/final/PC_prs_phoQ_no_abels.png", dpi=300, bbox_inches='tight')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
plt.rcParams["font.family"] = "Nimbus Roman"

# --- Define combined manual label order and alt labels for strain PC ---
manual_label_order = [
    # prs/phoQ
    'prs A114V', 'prs R79C', 'prs A76V', 'prs A11S', 'prs I151T',
    'phoQ G39C', 'phoQ IS2 insertion', 'phoQ small_indel', 'phoQ IS5 insertion',
    # other
    'waaF D13E', 'cysS Y298S', 'mrcB|mrcB T|T702|657S|S', 'pheS A14P',
    'relB P45S', 'yjbB L14Q', 'rbsA S157R', 'ygfB P184Q', 'aceK V471G',
    'yicC mobile_element_insertion', 'corA H75N', 'glnH A17V', 'pitA T184T',
    'corA L50Q', 'mgtL/mgtA small_indel', 'nadK T200P', 'eutH P393Q',
    'manZ/yobD snp_intergenic', 'cstA A392S', 'pgaC I94I', 'cra S18N',
    'yhfG/ppiA snp_intergenic', 'pgaC I81L', 'yaaA I52T',
    'yhaC/rnpB snp_intergenic', 'marR small_indel', 'abgT L394F', 'phoP R67H',
    'yaaA S53I', 'ytfJ A73E', 'mgtT Y23S', 'marC/marR small_indel',
    'marR Q117*', 'insH1/mmuP snp_intergenic', 'yfbS A38S',
    'insH5/lomR snp_intergenic', 'obgE Q71K', 'rapA W549*', 'gsk F3I',
    'rlmB D59E', 'yiiG/frvR snp_intergenic', 'bamD P32Q', 'bioH S225C'
]

alt_labels = {
    'prs A114V': 'Prs A114V',
    'prs R79C': 'Prs R79C',
    'prs A76V': 'Prs A76V',
    'prs A11S': 'Prs A11S',
    'prs I151T': 'Prs I151T',
    'phoQ G39C': 'PhoQ G39C',
    'phoQ IS2 insertion': r'$\it{phoQ}$ IS2 insertion',
    'phoQ small_indel': r'$\it{phoQ}$ indel',
    'phoQ IS5 insertion': r'$\it{phoQ}$ IS5 insertion',
    'waaF D13E': 'WaaF D13E',
    'cysS Y298S': 'CysS Y298S',
    'mrcB|mrcB T|T702|657S|S': 'MrcB T702S',
    'pheS A14P': 'PheS A14P',
    'relB P45S': 'RelB P45S',
    'yjbB L14Q': 'YjbB L14Q',
    'rbsA S157R': 'RbsA S157R',
    'ygfB P184Q': 'YgfB P184Q',
    'aceK V471G': 'AceK V471G',
    'yicC mobile_element_insertion': r'$\it{yicC}$ IS insertion',
    'corA H75N': 'CorA H75N',
    'glnH A17V': 'GlnH A17V',
    'pitA T184T': 'PitA T184T',
    'corA L50Q': 'CorA L50Q',
    'mgtL/mgtA small_indel': r'$\it{mgtL/mgtA}$ indel',
    'nadK T200P': 'NadK T200P',
    'eutH P393Q': 'EutH P393Q',
    'manZ/yobD snp_intergenic': r'$\it{manZ/yobD}$ snp intergenic',
    'cstA A392S': 'CstA A392S',
    'pgaC I94I': 'PgaC I94I',
    'cra S18N': 'Cra S18N',
    'yhfG/ppiA snp_intergenic': r'$\it{yhfG/ppiA}$ snp intergenic',
    'pgaC I81L': 'PgaC I81L',
    'yaaA I52T': 'YaaA I52T',
    'yhaC/rnpB snp_intergenic': r'$\it{yhaC/rnpB}$ snp intergenic',
    'marR small_indel': r'$\it{marR}$ indel',
    'abgT L394F': 'AbgT L394F',
    'phoP R67H': 'PhoP R67H',
    'yaaA S53I': 'YaaA S53I',
    'ytfJ A73E': 'YtfJ A73E',
    'mgtT Y23S': 'MgtT Y23S',
    'marC/marR small_indel': r'$\it{marC/marR}$ indel',
    'marR Q117*': 'MarR Q117*',
    'insH1/mmuP snp_intergenic': r'$\it{insH1/mmuP}$ snp intergenic',
    'yfbS A38S': 'YfbS A38S',
    'insH5/lomR snp_intergenic': r'$\it{insH5/lomR}$ snp intergenic',
    'obgE Q71K': 'ObgE Q71K',
    'rapA W549*': 'RapA W549*',
    'gsk F3I': 'Gsk F3I',
    'rlmB D59E': 'RlmB D59E',
    'yiiG/frvR snp_intergenic': r'$\it{yiiG/frvR}$ snp intergenic',
    'bamD P32Q': 'BamD P32Q',
    'bioH S225C': 'BioH S225C'
}


# --- Colormap for strain PC (Greens) ---
standard_map = plt.cm.get_cmap('Greens')
new_colors = standard_map(np.linspace(0, 1, 256))
new_colors[0] = np.array([1, 1, 1, 1])  # Set zero values to white
custom_map = mcolors.ListedColormap(new_colors)

# --- Ensure 'Pop' exists ---
for idx, df in enumerate(af):
    if 'Pop' not in df.columns:
        df['Pop'] = idx + 1

# --- Map label/Pop to final freq ---
freq_map = {
    (row['label'], row['Pop']): row['freq'][-1] if isinstance(row['freq'], (list, np.ndarray)) else row['freq']
    for df in af for _, row in df.iterrows()
}

# --- Add 'last_freq' to all rows ---
for df in af:
    df['last_freq'] = df.apply(lambda row: freq_map.get((row['label'], row['Pop']), 0), axis=1)

# --- Combine all dataframes ---
all_data = pd.concat(af, ignore_index=True)

# ✅ Only include rows present in manual_label_order
filtered_data = all_data[all_data['label'].isin(manual_label_order)].copy()

# --- Pivot table ---
heatmap_data = filtered_data.pivot_table(index='label', columns='Pop', values='last_freq', fill_value=0)
heatmap_data = heatmap_data.reindex(index=manual_label_order)

# --- Reindex culture columns ---
all_cultures = range(1, 11)
heatmap_data = heatmap_data.reindex(columns=all_cultures, fill_value=0)

# --- Prep for plotting ---
freq_mat = heatmap_data.values
timepoints = heatmap_data.columns
plot_labels = [alt_labels.get(label, label) for label in heatmap_data.index]
annot_matrix = np.where(freq_mat.T == 0, '', freq_mat.T.round(1).astype(str))

# --- Figure size ---
cell_width = 0.7
figsize = ((heatmap_data.shape[0] * cell_width), 7)

# --- Plot ---
fig, ax = plt.subplots(figsize=figsize)
sns.heatmap(
    freq_mat.T,
    yticklabels=timepoints,
    xticklabels=plot_labels,
    cmap=custom_map,
    vmin=0,
    vmax=1,
    ax=ax,
    annot=annot_matrix,
    fmt="",
    annot_kws={'size': 20, 'ha': 'center', 'va': 'center'},
    cbar=False,
    linewidths=0.5,
    linecolor='lightgrey'
)

# --- Axis styling ---
ax.set_ylabel('Culture Number', fontsize=25)
ax.set_xlabel('', fontsize=25)
ax.tick_params(axis='both', which='major', labelsize=15, length=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor', fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
    spine.set_color("black")

plt.subplots_adjust(bottom=0.1)
plt.tight_layout()

# --- Save figure ---
fig.savefig(PROJECT_PATH / "figures/final/PC_full.png", dpi=300, bbox_inches='tight')
plt.show()
